# Compare solar elevation: NSRDB vs CYL_GHI

Load one station from the merged NSRDB files and the matching CYL_GHI CSV, then plot solar elevation over the configured interval. NSRDB elevation is read from `solar_elev` or derived as `90 - Solar Zenith Angle`; CYL_GHI uses `sun_elev`. Timestamps are aligned before plotting.

Edit the paths, station code, and date range in the configuration cell below. The default paths follow `NoteBooks/data_preprocess.ipynb`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Data root containing NSRDB/ and CYL_GHI/.
# For Colab, mount Drive first and set this to your MyDrive directory.
BASE_DIR = Path.home() / 'MyDrive'
NSRDB_MERGED_DIR = BASE_DIR / 'NSRDB' / 'merged_files2'
CYL_GHI_DIR = BASE_DIR / 'CYL_GHI'

STATION_CODE = 'YOUR_STATION_CODE'  # e.g. the matching station_code in CYL_geo/CyL_geo.csv
START = '2017-06-01 00:00'
END = '2017-06-07 23:59'

NSRDB_FILE = NSRDB_MERGED_DIR / f'{STATION_CODE}_merged.csv'
# Resolve the raw CYL_GHI CSV by station-code prefix (same convention as data_preprocess.ipynb).
CYL_MATCHES = sorted(CYL_GHI_DIR.glob(f'{STATION_CODE}*.csv'))
if len(CYL_MATCHES) != 1:
    raise FileNotFoundError(f'Expected one CYL_GHI CSV starting with {STATION_CODE!r} in {CYL_GHI_DIR}; found: {CYL_MATCHES}')
CYL_FILE = CYL_MATCHES[0]
print('NSRDB:', NSRDB_FILE)
print('CYL_GHI:', CYL_FILE)

In [ ]:
def load_timestamp(df, label):
    if 'timestamp' in df.columns:
        ts = pd.to_datetime(df['timestamp'], errors='coerce')
    else:
        date_cols = ['Year', 'Month', 'Day', 'Hour', 'Minute']
        missing = [c for c in date_cols if c not in df.columns]
        if missing:
            raise KeyError(f'{label}: no timestamp column; missing date fields: {missing}')
        ts = pd.to_datetime(df[date_cols].rename(columns={
            'Year': 'year', 'Month': 'month', 'Day': 'day',
            'Hour': 'hour', 'Minute': 'minute'
        }), errors='coerce')
    return ts

if not NSRDB_FILE.exists():
    raise FileNotFoundError(f'NSRDB merged file not found: {NSRDB_FILE}')
if not CYL_FILE.exists():

nsrdb = pd.read_csv(NSRDB_FILE)
cyl = pd.read_csv(CYL_FILE)

if 'solar_elev' in nsrdb.columns:
    nsrdb_elev = pd.to_numeric(nsrdb['solar_elev'], errors='coerce')
elif 'Solar Zenith Angle' in nsrdb.columns:
    nsrdb_elev = 90 - pd.to_numeric(nsrdb['Solar Zenith Angle'], errors='coerce')
else:
    raise KeyError(f'No solar_elev or Solar Zenith Angle in {NSRDB_FILE}')

if 'sun_elev' not in cyl.columns:
    raise KeyError(f'No sun_elev column in {CYL_FILE}; columns are {list(cyl.columns)}')

nsrdb_plot = pd.DataFrame({'timestamp': load_timestamp(nsrdb, 'NSRDB'), 'NSRDB solar_elevation': nsrdb_elev})
cyl_plot = pd.DataFrame({'timestamp': load_timestamp(cyl, 'CYL_GHI'), 'CYL_GHI sun_elev': pd.to_numeric(cyl['sun_elev'], errors='coerce')})
nsrdb_plot = nsrdb_plot.dropna(subset=['timestamp']).drop_duplicates('timestamp').set_index('timestamp').sort_index()
cyl_plot = cyl_plot.dropna(subset=['timestamp']).drop_duplicates('timestamp').set_index('timestamp').sort_index()

start, end = pd.Timestamp(START), pd.Timestamp(END)
nsrdb_plot = nsrdb_plot.loc[start:end]
cyl_plot = cyl_plot.loc[start:end]
comparison = nsrdb_plot.join(cyl_plot, how='outer').sort_index()
print(f'NSRDB rows in range: {len(nsrdb_plot):,}; CYL_GHI rows in range: {len(cyl_plot):,}')
print(f'Overlapping timestamps: {nsrdb_plot.index.intersection(cyl_plot.index).size:,}')
comparison.head()

In [ ]:
ax = comparison.plot(figsize=(14, 5), marker='.', markersize=3, linewidth=1)
ax.set_title(f'Solar elevation comparison — {STATION_CODE} ({START} to {END})')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Solar elevation (degrees)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()